<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. It will feel more gritty than OpenAI Agents SDK or CrewAI.
            We will fix that by Day 3!
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [3]:
llm = ChatOpenAI(model="gpt-5.4-mini")
message = "Tell me something interesting about Astral projection"
reply = llm.invoke(message)
reply.content

'Astral projection is interesting because it sits at the crossroads of spirituality, psychology, and neuroscience.\n\nOne especially fascinating idea is that many people across different cultures report very similar experiences: feeling a separation from the body, floating above it, or traveling to another realm. Researchers think these experiences may sometimes be linked to the brain’s sense of body ownership and spatial awareness, which can be altered by dreaming, meditation, trauma, or certain neurological conditions.\n\nSo whether you view astral projection as a literal out-of-body journey or as a powerful altered state of consciousness, it’s a great example of how strange and flexible human perception can be.\n\nIf you want, I can also tell you:\n- the history of astral projection,\n- how it differs from lucid dreaming,\n- or the science behind out-of-body experiences.'

In [28]:
@tool
def getWeather(city: str) -> str:
  """Get the weather of a given city and Guess the temprature"""
  fake_weather = {"New York": "sunny", "Los Angeles": "cloudy", "Florida": "hot", "Chicago": "rainy"}
  return fake_weather.get(city, "unknown")

# print("name:", getWeather.name)
# print("description:", getWeather.description)
# print("args:", getWeather.args)
print(getWeather.invoke("Florida"))

hot


In [29]:
llm_with_tools = llm.bind_tools([getWeather])
reply = llm_with_tools.invoke("What is the weather in New York?")
print(reply.tool_calls)

[{'name': 'getWeather', 'args': {'city': 'New York'}, 'id': 'call_cw8Zl1OQunaAxY5uyuwd2lIq', 'type': 'tool_call'}]


In [37]:
chat = [HumanMessage("What is the Weather in New York?")]
llm_reply = llm_with_tools.invoke(chat)
chat.append(llm_reply)

for call in llm_reply.tool_calls:
    if call["name"] == "getWeather":
        result = getWeather.invoke(call["args"])
        chat.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke(chat)
print(final.content)


The weather in New York is sunny.


In [38]:
class Company(BaseModel):
    city: str = Field(description="The city name")
    weather: str = Field(description="The weather of the city")

structure_output_llm = llm.with_structured_output(Company)

getCityWeather = structure_output_llm.invoke("What is the weather in New York")
print(getCityWeather)


city='New York' weather="I don't have access to live weather data right now."
